# 04 — Xuất data mới sang Kaggle / nhập checkpoint về Drive (Colab CPU)

**Chạy notebook này trước trên Colab CPU → Run all.** Giữ KAGGLE_ACCOUNTS=2
nếu dùng hai tài khoản T4 x2; một tài khoản thì chọn 1 ngay trước lần xuất đầu.
Secret `KAGGLE_JSON` là của người xuất, bật Notebook access. Không gửi key vào chat.

Nhận batch **10/10 nguồn đã hoàn tất 03**. Giữ model/runtime và text hash cũ.
Chọn text thiếu bằng lookup SQL; không nạp weights, không đọc vector để kiểm kê.
Mỗi input part chọn xong lưu draft map + missing-text task + journal trên Drive.
Khi đổi runtime, phần đã có journal được bỏ qua; chỉ input part chưa commit mới đọc lại.

Gói Kaggle chứa text thiếu, không copy vector cũ. Hai GPU được phân công theo
hash để text trùng luôn vào cùng GPU và dùng lại vector vừa tạo. Mỗi lượt
đặt trần 16 GB kết quả/acc; GPU có deadline để kịp lưu Outputs trước khi runtime ngắt.
Có thể cần nhiều lượt. Đây là embedding child data mới, chưa ghép catalog/index,
chưa xử lý phần Qwen cũ còn thiếu và chưa xuất submission.

Sau GPU: điền `KAGGLE_NOTEBOOK_OUTPUT="owner/notebook,owner2/notebook2"` → Run all.
Nhập payload đã lưu và kiểm SHA/model/shape trước khi ghi coverage. Tự xuất lượt
còn thiếu; không đọc lại corpus đã chọn. Vector vào `data/model_cache`, progress
ở `data_staging/btc-new-71be0e12-20261019-v1/freeze_batch_v1/embeddings_v1`.
Các file tạm nằm trong runtime Colab, không tải corpus về máy cá nhân.
Luồng submit và active candidate được giữ nguyên. Chỉ một phiên CPU xuất/nhập cùng team.

In [ ]:
#@title 1. Colab CPU — chuẩn bị gói Kaggle cho batch 10 nguồn mới
import copy
import importlib
import json
import os
from pathlib import Path
import re
import subprocess
import sys
if not Path('/var/colab/hostname').is_file():
    raise RuntimeError('Notebook chuyển dữ liệu này chạy trên Colab CPU. Kaggle dùng 04_new_data_account_0 hoặc 1.')
print('NEW DATA: CPU export/import; không chạy lại 02/03, không nạp model weights.', flush=True)
KAGGLE_ACCOUNTS = 2 #@param {type:"integer"}
KAGGLE_NOTEBOOK_OUTPUT = "" #@param {type:"string"}
# Để trống để xuất gói. Sau GPU: owner/notebook[,owner2/notebook2] để nhập Outputs.
from google.colab import drive, auth, userdata
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/VietMedBridge')
relative = 'data_staging/btc-new-71be0e12-20261019-v1'
if not (PROJECT/relative/'freeze_batch_v1/freeze_batch.json').is_file():
    PROJECT = Path('/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge')
PROJECT = PROJECT.resolve()
lock = json.loads((PROJECT/relative/'intake_code_lock.json').read_text())
REPO_URL = 'https://github.com/Platypus27-coder/VietMedBridge.git'
if lock.get('repo_url') != REPO_URL or not re.fullmatch('[0-9a-f]{40}', lock.get('commit', '')):
    raise ValueError('Intake producer code lock không hợp lệ.')
try:
    credentials = json.loads(userdata.get('KAGGLE_JSON'))
    owner, api_key = credentials['username'], credentials['key']
    if not re.fullmatch('[A-Za-z0-9_-]+', owner) or not api_key:
        raise ValueError()
except Exception:
    raise RuntimeError('Thêm Colab Secret KAGGLE_JSON chứa nội dung kaggle.json và bật Notebook access.') from None
os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = owner, api_key
del credentials, api_key
CHECKOUT = Path('/content/VietMedBridge-new-kaggle')
WORK_DIR = Path('/content/vmb_new_kaggle_cpu')
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(CHECKOUT)], check=True)
if subprocess.check_output(['git', '-C', str(CHECKOUT), 'remote', 'get-url', 'origin'], text=True).strip() != REPO_URL:
    raise ValueError('Checkout thuộc repo khác.')
subprocess.run(['git', '-C', str(CHECKOUT), 'fetch', '--depth', '1', 'origin', lock['commit']], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', 'numpy', 'kaggle>=2.2.2', 'kagglehub'], cwd=CHECKOUT, check=True)
sys.path[:0] = [str(WORK_DIR), str(CHECKOUT/'src')]
importlib.invalidate_caches()
auth.authenticate_user()

In [ ]:
#@title 2. Kết nối manifest frozen và cache đã có — CPU, không mở vector payload
SOURCES = json.loads('{"new_data_kaggle.py": "\\"\\"\\"Sparse, resumable Kaggle handoff for the isolated frozen new-data batch.\\n\\nCPU selection opens each frozen part once, saves its draft references and only\\nexports uncached texts. Neither export nor import changes the submission pointer.\\n\\"\\"\\"\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\nimport argparse\\nimport json\\nimport os\\nimport re\\nimport signal\\nimport subprocess\\nimport sys\\nimport time\\n\\nimport numpy as np\\nimport pyarrow as pa\\nimport pyarrow.parquet as pq\\n\\nfrom vietmedbridge.artifacts import atomic_json, digest_json, publish_file, read_json, sha256_file, verify_file, utc_now\\nfrom vietmedbridge.content_embeddings import ContentBlocks, checked, seal, key\\nfrom vietmedbridge.embeddings import validate_vectors\\nfrom vietmedbridge.portable_embeddings import kaggle_assignment, validate_encoder, relative_path\\nfrom vietmedbridge.scale_vectors import bound\\ntry:\\n    from .new_data_embeddings import NewDataEmbeddings, LazyEncoder, embedding_contract, FAMILIES\\n    from .new_data_cache_lookup import ResumableContentBlocks\\nexcept ImportError:\\n    from new_data_embeddings import NewDataEmbeddings, LazyEncoder, embedding_contract, FAMILIES\\n    from new_data_cache_lookup import ResumableContentBlocks\\n\\nSOURCE_FILES = (\'new_data_kaggle.py\', \'new_data_embeddings.py\', \'new_data_cache_lookup.py\', \'new_data_drive_api.py\')\\nTASK_ROWS = 512\\n\\n\\ndef lookup(blocks, keys):\\n    known = {}\\n    keys = list(dict.fromkeys(keys))\\n    for start in range(0, len(keys), 900):\\n        selected = keys[start:start+900]\\n        rows = blocks.con.execute(\'SELECT key,path,sha,row FROM vectors WHERE key IN (\'+\\n                                  \',\'.join(\'?\' for _ in selected)+\')\', selected)\\n        known.update({k: {\'path\': p, \'sha256\': s, \'row\': r} for k, p, s, r in rows})\\n    return known\\n\\n\\ndef copied(source, target, checksum):\\n    if Path(target).is_file():\\n        verify_file(target, checksum)\\n    elif publish_file(source, target) != checksum:\\n        raise ValueError(\'Artifact changed during copy.\')\\n\\n\\nclass NewDataKaggle:\\n    def __init__(self, project, work, identities, *, accounts=2, gpu_count=2):\\n        self.assignment = kaggle_assignment(accounts=accounts, account_id=0, gpu_count=gpu_count)\\n        self.view = NewDataEmbeddings(project, work, identities)\\n        self.work = Path(work)\\n        self.team = {\'accounts\': accounts, \'gpu_count\': gpu_count}\\n        self.root = self.view.output / f\'kaggle-{accounts}x{gpu_count}\'\\n        self.identity = {\'schema\': \'new-data-selection-v1\', \'embedding_binding\': self.view.binding, \'team\': self.team}\\n        self.binding = digest_json(self.identity)\\n        contract = self.root / \'contract.json\'\\n        if contract.exists() and checked(read_json(contract)) != seal(self.identity):\\n            raise ValueError(\'Selection producer/team changed.\')\\n        atomic_json(contract, seal(self.identity))\\n        self.sources = {s[\'ordinal\']: s for s in self.view.sources}\\n\\n    def journals(self, family):\\n        result = {}\\n        for build in dict.fromkeys(s[\'build\'][\'build_run\'] for s in self.view.sources):\\n            path = self.root / \'selection\' / family / build / \'progress.json\'\\n            if not path.exists():\\n                continue\\n            saved = checked(read_json(path))\\n            if saved.get(\'binding\') != self.binding or saved.get(\'family\') != family:\\n                raise ValueError(\'Selection checkpoint belongs to another batch/model/team.\')\\n            for ordinal, record in saved[\'parts\'].items():\\n                source = self.sources[int(ordinal)]\\n                if (source[\'build\'][\'build_run\'] != build or record[\'input_sha256\'] != source[\'part\'][\'sha256\']\\n                        or record[\'ordinal\'] != int(ordinal) or record[\'rows\'] != source[\'part\'][\'rows\']):\\n                    raise ValueError(\'Selection checkpoint input binding changed.\')\\n                result[int(ordinal)] = record\\n        return result\\n\\n    def _save(self, family, source, selected):\\n        build = source[\'build\'][\'build_run\']\\n        record = {\'binding\': self.binding, \'family\': family, \'parts\': {\\n            str(k): v for k, v in selected.items() if self.sources[k][\'build\'][\'build_run\'] == build}}\\n        atomic_json(self.root / \'selection\' / family / build / \'progress.json\', seal(record))\\n\\n    def _map(self, family, source, refs):\\n        root, identity = self.view._view(family, source)\\n        config = root / \'config.json\'\\n        if config.exists() and read_json(config) != identity:\\n            raise ValueError(\'Embedding map producer changed.\')\\n        atomic_json(config, identity)\\n        part = source[\'part\']\\n        target = root / f\\"map-{part[\'start\']:012d}.parquet\\"\\n        local = self.work / \'map.parquet\'\\n        pq.write_table(pa.Table.from_pylist(refs), local, compression=\'zstd\')\\n        record = seal({\'signature\': digest_json(identity), \'start\': part[\'start\'], \'rows\': part[\'rows\'],\\n                       \'input_sha256\': part[\'sha256\'], \'path\': target.name, \'sha256\': publish_file(local, target)})\\n        atomic_json(root / f\\"map-{part[\'start\']:012d}.done.json\\", record)\\n        return record\\n\\n    def select(self, *, receipt_api=None, blocks_factory=ResumableContentBlocks):\\n        \\"\\"\\"Save draft refs and missing-text tasks before committing each input part.\\"\\"\\"\\n        for family in FAMILIES:\\n            selected = self.journals(family)\\n            pending = [s for s in self.view.sources if s[\'ordinal\'] not in selected]\\n            print(f\'SELECTION {family}: {len(selected)}/{len(self.sources)} parts saved; {len(pending)} new\', flush=True)\\n            if not pending:\\n                continue\\n            expected = self.view.identities[family]\\n            blocks = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\\n                                    self.work / family, receipt_api=receipt_api, progress=print)\\n            complete = self.view._journals(family)\\n            try:\\n                for source in pending:\\n                    part, ordinal = source[\'part\'], source[\'ordinal\']\\n                    path = bound(self.view.project, source[\'build\'][\'build_dir\']) / \'index_inputs\' / part[\'path\']\\n                    verify_file(path, part[\'sha256\'])\\n                    rows = pq.read_table(path, columns=[\'id\', \'text\']).to_pylist()\\n                    if len(rows) != part[\'rows\'] or any(r[\'id\'] != key(r[\'text\']) for r in rows):\\n                        raise ValueError(\'Frozen input row/hash mismatch.\')\\n                    known = lookup(blocks, [r[\'id\'] for r in rows])\\n                    draft = [{\'key\': r[\'id\'], **known.get(r[\'id\'], {\'path\': None, \'sha256\': None, \'row\': None})} for r in rows]\\n                    directory = self.root / \'selection\' / family / source[\'build\'][\'build_run\']\\n                    local = self.work / \'selection.parquet\'\\n                    schema = pa.schema([(\'key\', pa.string()), (\'path\', pa.string()), (\'sha256\', pa.string()), (\'row\', pa.int64())])\\n                    pq.write_table(pa.Table.from_pylist(draft, schema=schema), local, compression=\'zstd\')\\n                    draft_path = directory / f\'draft-{ordinal:06d}.parquet\'\\n                    draft_entry = {\'path\': draft_path.relative_to(self.root).as_posix(), \'sha256\': publish_file(local, draft_path)}\\n                    tasks, missing_rows = [], []\\n                    missing = [r for r in rows if r[\'id\'] not in known]\\n                    target = directory / f\'missing-{ordinal:06d}.parquet\'\\n                    for worker in range(self.assignment[\'workers\']):\\n                        group = [r for r in missing if int(r[\'id\'][:8], 16) % self.assignment[\'workers\'] == worker]\\n                        for offset in range(0, len(group), TASK_ROWS):\\n                            values = group[offset:offset+TASK_ROWS]\\n                            task_id = digest_json([self.binding, family, ordinal, worker, offset])\\n                            tasks.append({\'selection_id\': task_id, \'family\': family, \'ordinal\': ordinal, \'worker\': worker,\\n                                \'path\': target.relative_to(self.root).as_posix(), \'offset\': len(missing_rows),\\n                                \'rows\': len(values), \'keys_digest\': digest_json(sorted(r[\'id\'] for r in values))})\\n                            missing_rows.extend(values)\\n                    if missing_rows:\\n                        pq.write_table(pa.Table.from_pylist(missing_rows), local, compression=\'zstd\')\\n                        checksum = publish_file(local, target)\\n                        for task in tasks:\\n                            task[\'sha256\'] = checksum\\n                    selected[ordinal] = {\'ordinal\': ordinal, \'input_sha256\': part[\'sha256\'], \'rows\': part[\'rows\'],\\n                                         \'cached_rows\': len(rows)-len(missing), \'draft\': draft_entry, \'tasks\': tasks}\\n                    if not missing and self.view._key(source) not in complete:\\n                        complete[self.view._key(source)] = self._map(family, source, [known[r[\'id\']] for r in rows])\\n                        self.view._save(family, complete)\\n                    self._save(family, source, selected)\\n                    print(f\'SELECTION_CHECKPOINT {family} {len(selected)}/{len(self.sources)} | cached {len(rows)-len(missing)}, missing {len(missing)}\', flush=True)\\n                self.view._seal_views(family, complete)\\n            finally:\\n                blocks.close()\\n        return {f: self.journals(f) for f in FAMILIES}\\n\\n    def imported(self):\\n        done = set()\\n        for path in (self.root / \'imports\').glob(\'*.json\'):\\n            saved = checked(read_json(path))\\n            if saved.get(\'binding\') != self.binding:\\n                raise ValueError(\'Imported output binding changed.\')\\n            done.update(saved[\'selection_ids\'])\\n        return done\\n\\n    def complete_maps(self, family, ordinals, blocks):\\n        selected, completed = self.journals(family), self.view._journals(family)\\n        for ordinal in sorted(ordinals):\\n            source = self.sources[ordinal]\\n            if self.view._key(source) in completed:\\n                continue\\n            entry = selected[ordinal][\'draft\']\\n            path = bound(self.root, entry[\'path\'])\\n            verify_file(path, entry[\'sha256\'])\\n            draft = pq.read_table(path).to_pylist()\\n            if len(draft) != source[\'part\'][\'rows\']:\\n                raise ValueError(\'Draft map row coverage changed.\')\\n            missing = [r[\'key\'] for r in draft if r[\'path\'] is None]\\n            known = lookup(blocks, missing)\\n            if all(k in known for k in missing):\\n                refs = [{k: r[k] for k in (\'path\', \'sha256\', \'row\')} if r[\'path\'] is not None else known[r[\'key\']] for r in draft]\\n                completed[self.view._key(source)] = self._map(family, source, refs)\\n                self.view._save(family, completed)\\n        self.view._seal_views(family, completed)\\n\\n    def export(self, checkout, *, config, runtime, code_commit, receipt_api=None,\\n               blocks_factory=ResumableContentBlocks, output_bytes=16_000_000_000):\\n        active = self.root/\'active_round.json\'\\n        if active.exists():\\n            saved = checked(read_json(active))\\n            if saved[\'binding\'] != self.binding:\\n                raise ValueError(\'Active Kaggle round binding changed.\')\\n            imported = {r[\'account_id\'] for p in (self.root/\'imports\').glob(\'*.json\')\\n                        if (r := checked(read_json(p))).get(\'job\') == saved[\'job\'] and \'account_id\' in r}\\n            if imported != set(range(self.team[\'accounts\'])):\\n                return None, {\'state\': \'WAITING_FOR_KAGGLE_OUTPUTS\', \'dataset_handle\': saved[\'dataset_handle\'],\\n                    \'imported_accounts\': sorted(imported), \'required_accounts\': list(range(self.team[\'accounts\'])),\\n                    \'instruction\': \'Use the existing dataset; import saved Outputs of the current round before starting another round.\'}\\n        selected = self.select(receipt_api=receipt_api, blocks_factory=blocks_factory)\\n        if type(output_bytes) is not int or not 0 < output_bytes <= 16_000_000_000:\\n            raise ValueError(\'Output budget must be between 1 and 16 GB per account.\')\\n        if not re.fullmatch(\'[0-9a-f]{40}\', code_commit):\\n            raise ValueError(\'Pin the full native producer commit.\')\\n        done = self.imported()\\n        used, tasks, remaining = [0]*self.team[\'accounts\'], [], {f: 0 for f in FAMILIES}\\n        assets = self.work/\'export-assets\'\\n        assets.mkdir(parents=True, exist_ok=True)\\n        for family in FAMILIES:\\n            pending = [task for entry in selected[family].values() for task in entry[\'tasks\'] if task[\'selection_id\'] not in done]\\n            remaining[family] = len(pending)\\n            if not pending:\\n                continue\\n            expected = self.view.identities[family]\\n            blocks = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\\n                                    self.work/family, receipt_api=receipt_api, progress=print)\\n            cache_only, affected = [], set()\\n            last_path, last_rows = None, None\\n            try:\\n                for task in pending:\\n                    account = task[\'worker\'] // self.team[\'gpu_count\']\\n                    estimate = task[\'rows\'] * (expected[\'dimension\']*4+1024)\\n                    if estimate > output_bytes:\\n                        raise ValueError(\'One task exceeds the round output budget.\')\\n                    if used[account]+estimate > output_bytes:\\n                        continue\\n                    path = bound(self.root, task[\'path\'])\\n                    if path != last_path:\\n                        verify_file(path, task[\'sha256\'])\\n                        last_rows = pq.read_table(path, columns=[\'id\', \'text\']).to_pylist()\\n                        last_path = path\\n                    rows = last_rows[task[\'offset\']:task[\'offset\']+task[\'rows\']]\\n                    if len(rows) != task[\'rows\'] or digest_json(sorted(r[\'id\'] for r in rows)) != task[\'keys_digest\']:\\n                        raise ValueError(\'Saved missing-text task changed.\')\\n                    known = lookup(blocks, [r[\'id\'] for r in rows])\\n                    rows = [r for r in rows if r[\'id\'] not in known]\\n                    if not rows:\\n                        cache_only.append(task[\'selection_id\'])\\n                        affected.add(task[\'ordinal\'])\\n                        remaining[family] -= 1\\n                        continue\\n                    estimate = len(rows) * (expected[\'dimension\']*4+1024)\\n                    used[account] += estimate\\n                    asset = \'task-\'+task[\'selection_id\']+\'.parquet\'\\n                    pq.write_table(pa.Table.from_pylist(rows), assets/asset, compression=\'zstd\')\\n                    tasks.append({**task, \'id\': task[\'selection_id\'], \'asset\': asset, \'rows\': len(rows),\\n                        \'sha256\': sha256_file(assets/asset), \'keys_digest\': digest_json(sorted(r[\'id\'] for r in rows)),\\n                        \'estimated_output_bytes\': estimate})\\n                if cache_only:\\n                    self.complete_maps(family, affected, blocks)\\n                    atomic_json(self.root/\'imports\'/(\'cache-\'+digest_json(cache_only)+\'.json\'), seal({\\n                        \'binding\': self.binding, \'selection_ids\': cache_only, \'state\': \'REUSED_NEWLY_AVAILABLE_CACHE\'}))\\n            finally:\\n                blocks.close()\\n        if not tasks:\\n            return None, {\'state\': \'NO_PENDING_KAGGLE_TASKS\', \'coverage\': self.coverage()}\\n        model_code = {n: sha256_file(Path(checkout)/\'src/vietmedbridge\'/n) for n in (\'retrieval_models.py\', \'qwen_models.py\')}\\n        scripts = Path(__file__).resolve().parent\\n        workers = {n: sha256_file(scripts/n) for n in SOURCE_FILES}\\n        job = seal({\'schema\': \'new-data-kaggle-v1\', \'selection_binding\': self.binding, \'batch\': self.view.batch[\'manifest_sha256\'],\\n            \'team\': self.team, \'code_commit\': code_commit, \'repo_url\': \'https://github.com/Platypus27-coder/VietMedBridge.git\',\\n            \'models\': self.view.identities, \'config\': config, \'runtime\': runtime, \'model_code\': model_code,\\n            \'worker_sources\': workers, \'tasks\': tasks, \'remaining_tasks_before_round\': remaining,\\n            \'scheduled_bytes_per_account\': used, \'scope\': \'New frozen child embeddings only; no submission inference.\'})\\n        target = self.work / \'exports\' / job[\'manifest_sha256\'][:16]\\n        target.mkdir(parents=True, exist_ok=True)\\n        for i, task in enumerate(tasks, 1):\\n            copied(assets/task[\'asset\'], target/task[\'asset\'], task[\'sha256\'])\\n            if i % 100 == 0:\\n                print(f\'KAGGLE_EXPORT {i}/{len(tasks)} missing-text tasks\', flush=True)\\n        for name, checksum in workers.items():\\n            copied(scripts/name, target/name, checksum)\\n        atomic_json(target/\'job.json\', job)\\n        atomic_json(self.root/\'exports\'/(\'job-\'+job[\'manifest_sha256\']+\'.json\'), job)\\n        return target, job\\n\\n    def coverage(self):\\n        return {f: {\'completed_parts\': len(self.view._journals(f)), \'total_parts\': len(self.sources)} for f in FAMILIES}\\n\\n    def install(self, output, *, receipt_api=None, blocks_factory=ResumableContentBlocks):\\n        output = Path(output)\\n        job = checked(read_json(output/\'job.json\'))\\n        saved_job = checked(read_json(self.root/\'exports\'/(\'job-\'+job[\'manifest_sha256\']+\'.json\')))\\n        result = checked(read_json(output/\'new-data-result.json\'))\\n        if job != saved_job or job[\'selection_binding\'] != self.binding or job[\'models\'] != self.view.identities or result[\'job\'] != job[\'manifest_sha256\']:\\n            raise ValueError(\'Kaggle output differs from the frozen export/model contract.\')\\n        prior_path = self.root/\'imports\'/(result[\'manifest_sha256\']+\'.json\')\\n        if prior_path.exists():\\n            prior = checked(read_json(prior_path))\\n            if (prior.get(\'binding\') != self.binding or prior.get(\'job\') != job[\'manifest_sha256\']\\n                    or prior.get(\'result\') != result[\'manifest_sha256\']):\\n                raise ValueError(\'Committed import receipt binding changed.\')\\n            return seal({**{k:v for k,v in prior.items() if k != \'manifest_sha256\'},\\n                         \'coverage\': self.coverage(), \'payloads_reopened\': False})\\n        assignment = kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=result[\'account_id\'], gpu_count=job[\'team\'][\'gpu_count\'])\\n        tasks = {t[\'id\']: t for t in job[\'tasks\']}\\n        expected_keys, affected, completed_ids = {}, {f: set() for f in FAMILIES}, []\\n        for receipt in result[\'tasks\']:\\n            task = tasks[receipt[\'task\']]\\n            if task[\'worker\'] not in assignment[\'worker_ids\'] or receipt[\'task\'] in completed_ids:\\n                raise ValueError(\'Output contains duplicate/unassigned tasks.\')\\n            keys = receipt[\'keys\']\\n            if len(keys) != task[\'rows\'] or len(set(keys)) != len(keys) or digest_json(sorted(keys)) != task[\'keys_digest\']:\\n                raise ValueError(\'Output task key coverage changed.\')\\n            expected_keys.setdefault(task[\'family\'], set()).update(keys)\\n            completed_ids.append(receipt[\'task\'])\\n            affected[task[\'family\']].add(task[\'ordinal\'])\\n        # Validate all output files before publishing any coverage receipt.\\n        blocks_by_family = {f: [] for f in FAMILIES}\\n        available = {f: set() for f in FAMILIES}\\n        for block in result[\'blocks\']:\\n            checked(block)\\n            family = next((f for f in FAMILIES if block[\'identity\'] == {\\n                **self.view.identities[f], \'format\': \'content-float32-l2-v1\'}), None)\\n            if family is None or not set(block[\'keys\']) <= expected_keys.get(family, set()) or len(block[\'keys\']) != block[\'rows\']:\\n                raise ValueError(\'Unexpected vector block producer/keys.\')\\n            path = bound(output, block[\'path\'])\\n            verify_file(path, block[\'sha256\'])\\n            validate_vectors(np.load(path, mmap_mode=\'r\', allow_pickle=False), block[\'rows\'], block[\'identity\'][\'dimension\'])\\n            if path.parent.relative_to(output).as_posix() != \'model_cache/\'+digest_json(block[\'identity\'])[:24]:\\n                raise ValueError(\'Output vector path/producer namespace changed.\')\\n            blocks_by_family[family].append(block)\\n            available[family].update(block[\'keys\'])\\n        if any(not keys <= available[f] for f, keys in expected_keys.items()):\\n            raise ValueError(\'Output tasks are missing saved vector payloads.\')\\n        for family in FAMILIES:\\n            if not affected[family]:\\n                continue\\n            expected = self.view.identities[family]\\n            blocks = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\\n                                    self.work/family, receipt_api=receipt_api, progress=print)\\n            try:\\n                for block in blocks_by_family[family]:\\n                    path = bound(self.view.data, block[\'path\'])\\n                    if path.exists():\\n                        verify_file(path, block[\'sha256\'])\\n                    else:\\n                        copied(bound(output, block[\'path\']), path, block[\'sha256\'])\\n                    blocks.save_block(block[\'keys\'], path, block[\'sha256\'])\\n                self.complete_maps(family, affected[family], blocks)\\n                if getattr(blocks, \'_snapshot_dirty\', False):\\n                    blocks._snapshot()\\n            finally:\\n                blocks.close()\\n        report = seal({\'state\': \'NEW_DATA_KAGGLE_IMPORTED\', \'binding\': self.binding, \'job\': job[\'manifest_sha256\'],\\n            \'result\': result[\'manifest_sha256\'], \'account_id\': result[\'account_id\'], \'selection_ids\': completed_ids,\\n            \'coverage\': self.coverage(), \'updated_at\': utc_now()})\\n        atomic_json(self.root/\'imports\'/(result[\'manifest_sha256\']+\'.json\'), report)\\n        return report\\n\\n\\ndef validate_job(directory, checkout=None):\\n    directory = Path(directory)\\n    job = checked(read_json(directory/\'job.json\'))\\n    if (job.get(\'schema\') != \'new-data-kaggle-v1\' or job.get(\'repo_url\') != \'https://github.com/Platypus27-coder/VietMedBridge.git\'\\n            or not re.fullmatch(\'[0-9a-f]{40}\', job.get(\'code_commit\', \'\'))):\\n        raise ValueError(\'Invalid new-data Kaggle job.\')\\n    kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=0, gpu_count=job[\'team\'][\'gpu_count\'])\\n    if set(job[\'worker_sources\']) != set(SOURCE_FILES) or set(job[\'models\']) != set(FAMILIES):\\n        raise ValueError(\'Missing model/worker producer contract.\')\\n    for name, checksum in job[\'worker_sources\'].items():\\n        verify_file(directory/name, checksum)\\n    seen = set()\\n    for task in job[\'tasks\']:\\n        if (task[\'id\'] in seen or task[\'id\'] != task[\'selection_id\'] or task[\'family\'] not in FAMILIES\\n                or type(task[\'worker\']) is not int or not 0 <= task[\'worker\'] < job[\'team\'][\'accounts\']*job[\'team\'][\'gpu_count\']\\n                or type(task[\'rows\']) is not int or not 0 < task[\'rows\'] <= TASK_ROWS\\n                or relative_path(task[\'asset\']) != \'task-\'+task[\'id\']+\'.parquet\'):\\n            raise ValueError(\'Invalid sparse task assignment.\')\\n        seen.add(task[\'id\'])\\n    if checkout is not None:\\n        for name, checksum in job[\'model_code\'].items():\\n            if name not in (\'retrieval_models.py\', \'qwen_models.py\'):\\n                raise ValueError(\'Unexpected model code file.\')\\n            verify_file(Path(checkout)/\'src/vietmedbridge\'/name, checksum)\\n        for family in FAMILIES:\\n            validate_encoder(job, family, job[\'models\'][family][\'encoder\'], job[\'models\'][family][\'dimension\'])\\n    return job\\n\\n\\ndef download_outputs(api, handle, work, *, project=None):\\n    \\"\\"\\"Page through the new-data manifest first, before downloading vectors.\\"\\"\\"\\n    from vietmedbridge.kaggle_outputs import _download_pages, output_handles\\n    output_handles(handle)\\n    status = api.kernels_status(handle)\\n    status = status.get(\'status\') if isinstance(status, dict) else status.status\\n    status = getattr(status, \'name\', str(status)).upper()\\n    if status != \'COMPLETE\':\\n        raise RuntimeError(f\'{handle}: status={status}; wait for the saved version to finish.\')\\n    base = Path(work)/\'downloads\'/digest_json(handle)[:16]\\n    probe = base/(\'probe-\'+str(time.time_ns()))\\n    probe.mkdir(parents=True, exist_ok=True)\\n    paths = _download_pages(api, handle, probe, file_pattern=r\'(^|/)new-data-result[.]json$\', force=True)\\n    manifests = [p for p in paths if p.name == \'new-data-result.json\']\\n    if len(manifests) != 1:\\n        raise RuntimeError(f\'{handle}: expected one new-data-result.json, found {len(manifests)}; choose the saved new-data worker version.\')\\n    manifest = checked(read_json(manifests[0]))\\n    relative = manifests[0].relative_to(probe)\\n    if project is not None:\\n        from_parent = Path(project)/\'data_staging\'/\'btc-new-71be0e12-20261019-v1\'/\'freeze_batch_v1\'/\'embeddings_v1\'\\n        for receipt in from_parent.glob(\'kaggle-*/imports/\'+manifest[\'manifest_sha256\']+\'.json\'):\\n            prior = checked(read_json(receipt))\\n            if prior.get(\'result\') != manifest[\'manifest_sha256\'] or prior.get(\'job\') != manifest[\'job\']:\\n                raise ValueError(\'Previously imported output receipt changed.\')\\n            job = checked(read_json(receipt.parent.parent/\'exports\'/(\'job-\'+manifest[\'job\']+\'.json\')))\\n            atomic_json(manifests[0].parent/\'job.json\', job)\\n            print(\'OUTPUT_ALREADY_IMPORTED: sealed receipt reused; no vector download\', flush=True)\\n            return manifests[0].parent\\n    destination = base/manifest[\'manifest_sha256\'][:16]\\n    destination.mkdir(parents=True, exist_ok=True)\\n    paths = _download_pages(api, handle, destination, force=False, quiet=False)\\n    target = destination/relative\\n    if target not in paths or checked(read_json(target)) != manifest:\\n        raise ValueError(\'Saved Kaggle Outputs changed after preflight; select a fixed version.\')\\n    return target.parent\\n\\n\\ndef upload_job(flow, target, job, owner):\\n    \\"\\"\\"A unique private dataset, with an upload pointer retained on Drive.\\"\\"\\"\\n    import kagglehub\\n    if not re.fullmatch(\'[A-Za-z0-9_-]+\', owner):\\n        raise ValueError(\'Invalid Kaggle owner.\')\\n    pointer = flow.root/\'uploads\'/(owner+\'-\'+job[\'manifest_sha256\']+\'.json\')\\n    if pointer.exists():\\n        saved = read_json(pointer)\\n        if saved[\'job\'] != job[\'manifest_sha256\']:\\n            raise ValueError(\'Upload pointer belongs to another job.\')\\n        handle = saved[\'dataset_handle\']\\n    else:\\n        handle = owner+\'/vmb-new-\'+job[\'manifest_sha256\'][:12]+\'-\'+str(time.time_ns())\\n        atomic_json(pointer, {\'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle, \'state\': \'UPLOAD_PLANNED\'})\\n        saved = read_json(pointer)\\n    if saved[\'state\'] == \'UPLOAD_PLANNED\':\\n        # Deterministic pointer survives a failed upload; never choose a second handle.\\n        kagglehub.dataset_upload(handle, str(target))\\n        atomic_json(pointer, {\'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle, \'state\': \'VERIFY_PENDING\'})\\n    remote = None\\n    for attempt in range(6):\\n        try:\\n            remote = Path(kagglehub.dataset_download(handle, path=\'job.json\', force_download=True))\\n            break\\n        except Exception:\\n            if attempt == 5:\\n                raise RuntimeError(\'Dataset is processing or inaccessible; rerun to verify the same upload.\') from None\\n            time.sleep(min(30, 5*(attempt+1)))\\n    if read_json(remote) != job:\\n        raise ValueError(\'Uploaded Kaggle job header differs; do not start GPU workers.\')\\n    atomic_json(pointer, {\'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle, \'state\': \'VERIFIED\'})\\n    atomic_json(flow.root/\'active_round.json\', seal({\'binding\': flow.binding, \'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle}))\\n    print(\'NEW_DATASET_READY:\', \'https://www.kaggle.com/datasets/\'+handle, flush=True)\\n    print(\'Team:\', job[\'team\'], \'| scheduled missing-text tasks:\', len(job[\'tasks\']), flush=True)\\n    return handle\\n\\n\\ndef run_worker(directory, output, work, *, worker_id, factories, deadline, batch_sizes=None):\\n    \\"\\"\\"Each GPU owns every occurrence of a given text hash, avoiding duplicate encoding.\\"\\"\\"\\n    directory, output, work = Path(directory), Path(output), Path(work)\\n    job = validate_job(directory)\\n    if type(worker_id) is not int or not 0 <= worker_id < job[\'team\'][\'accounts\']*job[\'team\'][\'gpu_count\']:\\n        raise ValueError(\'Worker ID is outside this job.\')\\n    output.mkdir(parents=True, exist_ok=True)\\n    if (output/\'job.json\').exists() and read_json(output/\'job.json\') != job:\\n        raise ValueError(\'Working output contains another job. Save its Outputs; start a fresh Kaggle version.\')\\n    atomic_json(output/\'job.json\', job)\\n    journal = output/f\'worker-{worker_id}.json\'\\n    completed = []\\n    if journal.exists():\\n        saved = checked(read_json(journal))\\n        if saved[\'job\'] != job[\'manifest_sha256\'] or saved[\'worker_id\'] != worker_id:\\n            raise ValueError(\'Worker journal belongs to another job/worker.\')\\n        completed = saved[\'tasks\']\\n    done = {r[\'task\'] for r in completed}\\n    if len(done) != len(completed) or not done <= {t[\'id\'] for t in job[\'tasks\'] if t[\'worker\'] == worker_id}:\\n        raise ValueError(\'Duplicate local task receipt.\')\\n    stop_reason = \'ASSIGNMENT_COMPLETE\'\\n    for family in FAMILIES:\\n        assigned = [t for t in job[\'tasks\'] if t[\'worker\'] == worker_id and t[\'family\'] == family and t[\'id\'] not in done]\\n        print(f\'GPU worker {worker_id}: {family}, {len(assigned)} missing-text tasks\', flush=True)\\n        if not assigned or time.monotonic() >= deadline:\\n            if assigned:\\n                stop_reason = \'SESSION_TIME_LIMIT\'\\n            continue\\n        lazy = LazyEncoder(job[\'models\'][family], factories[family])\\n        blocks = ContentBlocks(output, lazy, work/family)\\n        try:\\n            for task in assigned:\\n                if time.monotonic() >= deadline:\\n                    stop_reason = \'SESSION_TIME_LIMIT\'\\n                    break\\n                path = directory/task[\'asset\']\\n                verify_file(path, task[\'sha256\'])\\n                rows = pq.read_table(path, columns=[\'id\', \'text\']).to_pylist()\\n                keys = [r[\'id\'] for r in rows]\\n                workers = job[\'team\'][\'accounts\']*job[\'team\'][\'gpu_count\']\\n                if (len(rows) != task[\'rows\'] or digest_json(sorted(keys)) != task[\'keys_digest\']\\n                        or any(k != key(r[\'text\']) or int(k[:8], 16) % workers != worker_id for k, r in zip(keys, rows))):\\n                    raise ValueError(\'Task payload/hash/ownership changed.\')\\n                blocks.refs([r[\'text\'] for r in rows], batch_size=(batch_sizes or {\'bge\': 32, \'qwen\': 16})[family],\\n                            encode_missing=True, encode_block_rows=TASK_ROWS)\\n                completed.append({\'task\': task[\'id\'], \'keys\': keys})\\n                atomic_json(journal, seal({\'job\': job[\'manifest_sha256\'], \'worker_id\': worker_id, \'tasks\': completed}))\\n                print(f\'KAGGLE_CHECKPOINT worker {worker_id} | saved tasks {len(completed)} | {family} | /kaggle/working\', flush=True)\\n        finally:\\n            blocks.close()\\n            lazy.close()\\n    return seal({\'job\': job[\'manifest_sha256\'], \'worker_id\': worker_id, \'tasks\': completed, \'stop_reason\': stop_reason})\\n\\n\\ndef finalize(directory, output, account_id):\\n    job = validate_job(directory)\\n    output = Path(output)\\n    assignment = kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=account_id, gpu_count=job[\'team\'][\'gpu_count\'])\\n    root = output/\'result\'\\n    root.mkdir(parents=True, exist_ok=True)\\n    tasks = []\\n    for worker in assignment[\'worker_ids\']:\\n        path = root/f\'worker-{worker}.json\'\\n        if path.exists():\\n            saved = checked(read_json(path))\\n            if saved[\'job\'] != job[\'manifest_sha256\'] or saved[\'worker_id\'] != worker:\\n                raise ValueError(\'Worker journal belongs to another job/assignment.\')\\n            tasks.extend(saved[\'tasks\'])\\n    task_family = {t[\'id\']: t[\'family\'] for t in job[\'tasks\']}\\n    committed = {f: {k for task in tasks if task_family[task[\'task\']] == f for k in task[\'keys\']} for f in FAMILIES}\\n    blocks = {}\\n    for path in root.glob(\'model_cache/*/*.done.json\'):\\n        block = checked(read_json(path))\\n        family = next((f for f in FAMILIES if block[\'identity\'] == {**job[\'models\'][f], \'format\': \'content-float32-l2-v1\'}), None)\\n        if family is not None and set(block[\'keys\']) <= committed[family]:\\n            blocks[block[\'path\']] = block\\n    atomic_json(root/\'job.json\', job)\\n    result = seal({\'schema\': \'new-data-kaggle-result-v1\', \'job\': job[\'manifest_sha256\'],\\n        \'account_id\': account_id, \'tasks\': tasks, \'blocks\': list(blocks.values()), \'updated_at\': utc_now(),\\n        \'state\': \'ROUND_COMPLETE\' if len(tasks) == sum(t[\'worker\'] in assignment[\'worker_ids\'] for t in job[\'tasks\']) else \'CHECKPOINTED_PARTIAL\'})\\n    atomic_json(root/\'new-data-result.json\', result)\\n    return result\\n\\n\\ndef launch(directory, checkout, output, work, *, account_id=0, hours=10.0):\\n    \\"\\"\\"Fresh CUDA subprocesses, one model per T4; leave time to save Kaggle Outputs.\\"\\"\\"\\n    import torch\\n    from vietmedbridge.portable_embeddings import embedding_runtime\\n    job = validate_job(directory, checkout)\\n    if embedding_runtime() != job[\'runtime\'] or not torch.cuda.is_available() or torch.cuda.device_count() != job[\'team\'][\'gpu_count\']:\\n        raise RuntimeError(\'CUDA/runtime mismatch: select T4 x2 and use the locked CUDA packages.\')\\n    assignment = kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=account_id, gpu_count=job[\'team\'][\'gpu_count\'])\\n    if not 0 < hours <= 10.5:\\n        raise ValueError(\'Use a save deadline up to 10.5 hours, before the provider hard stop.\')\\n    deadline = time.monotonic()+hours*3600-180\\n    processes = []\\n    try:\\n        for local_gpu, worker in enumerate(assignment[\'worker_ids\']):\\n            env = os.environ.copy()\\n            env[\'CUDA_VISIBLE_DEVICES\'] = str(local_gpu)\\n            env[\'PYTHONPATH\'] = str(Path(checkout)/\'src\')+os.pathsep+str(Path(directory))\\n            command = [sys.executable, \'-u\', str(Path(directory)/\'new_data_kaggle.py\'), \'worker\', \'--job\', str(directory),\\n                       \'--output\', str(Path(output)/\'result\'), \'--work\', str(Path(work)/f\'worker-{worker}\'),\\n                       \'--worker-id\', str(worker), \'--deadline\', str(deadline)]\\n            processes.append(subprocess.Popen(command, env=env))\\n        while any(p.poll() is None for p in processes):\\n            if any(p.poll() not in (None, 0) for p in processes):\\n                raise RuntimeError(\'A GPU worker failed; outputs of completed tasks remain available.\')\\n            time.sleep(2)\\n        result = finalize(directory, output, account_id)\\n        if any(p.returncode != 0 for p in processes):\\n            raise RuntimeError(\'A GPU worker failed; inspect the preceding traceback.\')\\n        return result\\n    finally:\\n        for process in processes:\\n            if process.poll() is None:\\n                process.terminate()\\n        for process in processes:\\n            if process.poll() is None:\\n                try:\\n                    process.wait(timeout=30)\\n                except subprocess.TimeoutExpired:\\n                    process.kill()\\n                    process.wait(timeout=10)\\n        # Even an exception publishes a partial result for completed tasks.\\n        finalize(directory, output, account_id)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\'mode\', choices=[\'worker\', \'launch\'])\\n    parser.add_argument(\'--job\', required=True)\\n    parser.add_argument(\'--output\', required=True)\\n    parser.add_argument(\'--work\', required=True)\\n    parser.add_argument(\'--checkout\')\\n    parser.add_argument(\'--worker-id\', type=int)\\n    parser.add_argument(\'--deadline\', type=float)\\n    parser.add_argument(\'--account-id\', type=int, default=0)\\n    parser.add_argument(\'--hours\', type=float, default=10.0)\\n    args = parser.parse_args()\\n    if args.mode == \'launch\':\\n        result = launch(args.job, args.checkout, args.output, args.work, account_id=args.account_id, hours=args.hours)\\n    else:\\n        import torch\\n        from vietmedbridge.portable_embeddings import embedding_runtime\\n        from vietmedbridge.retrieval_models import TorchDenseEncoder\\n        from vietmedbridge.qwen_models import TorchQwenEncoder\\n        job = validate_job(args.job)\\n        if not torch.cuda.is_available() or embedding_runtime() != job[\'runtime\']:\\n            raise RuntimeError(\'GPU worker runtime differs from the paid vector producer.\')\\n        result = run_worker(args.job, args.output, args.work, worker_id=args.worker_id, deadline=args.deadline,\\n            factories={\'bge\': lambda: TorchDenseEncoder(job[\'config\'][\'dense\']), \'qwen\': lambda: TorchQwenEncoder(job[\'config\'][\'second_dense\'])})\\n    print(json.dumps(result, ensure_ascii=False, indent=2), flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "new_data_embeddings.py": "\\"\\"\\"Frozen-input and embedding-map contracts for the isolated ten-source batch.\\n\\nNative model code and content-cache keys stay unchanged. Only new input parts\\nare opened. A worker journal replaces repeated scans of old map payloads.\\n\\"\\"\\"\\nfrom pathlib import Path\\n\\n\\nfrom vietmedbridge.artifacts import (atomic_json, digest_json, publish_file,\\n    read_json, sha256_file, utc_now, verify_file)\\nfrom vietmedbridge.content_embeddings import checked, key, seal\\nfrom vietmedbridge.portable_embeddings import resources_relative, validate_encoder, validate_runtime_lock\\nfrom vietmedbridge.scale_vectors import bound\\n\\ntry:\\n    from .new_data_cache_lookup import ResumableContentBlocks\\nexcept ImportError:\\n    from new_data_cache_lookup import ResumableContentBlocks\\n\\nPARENT = \\"btc-new-71be0e12-20261019-v1\\"\\nFAMILIES = (\\"bge\\", \\"qwen\\")\\n\\n\\ndef load_new_batch(project):\\n    project = Path(project).resolve()\\n    root = project / \\"data_staging\\" / PARENT / \\"freeze_batch_v1\\"\\n    batch, contract = checked(read_json(root / \\"freeze_batch.json\\")), checked(read_json(root / \\"contract.json\\"))\\n    if (batch.get(\\"state\\") != \\"NEW_DATA_INPUTS_READY\\" or batch.get(\\"completed_builds\\") != 10\\n            or batch.get(\\"total_builds\\") != 10 or batch[\\"contract_sha256\\"] != contract[\\"manifest_sha256\\"]\\n            or len(batch[\\"builds\\"]) != 10 or len(contract[\\"sources\\"]) != 10):\\n        raise ValueError(\\"The complete ten-source frozen input handoff is required.\\")\\n    expected = {f\\"new-{i:05d}\\" for i in range(9)} | {\\"retained-71be0e12\\"}\\n    if {b[\\"build_run\\"] for b in batch[\\"builds\\"]} != expected:\\n        raise ValueError(\\"New-data batch source coverage changed.\\")\\n    sources, ordinal = [], 0\\n    for item, source in zip(batch[\\"builds\\"], contract[\\"sources\\"]):\\n        checked(item)\\n        if (item[\\"state\\"] != \\"COMPLETE\\" or item[\\"contract_sha256\\"] != contract[\\"manifest_sha256\\"]\\n                or any(item[k] != source[k] for k in (\\"build_run\\", \\"build_dir\\", \\"snapshot_sha256\\", \\"counts\\"))):\\n            raise ValueError(\\"Frozen source receipt differs from the batch contract.\\")\\n        build = bound(project, item[\\"build_dir\\"])\\n        inputs = checked(read_json(build / \\"index_inputs/units.json\\"))\\n        if (inputs.get(\\"state\\") != \\"COMPLETE\\" or inputs[\\"manifest_sha256\\"] != item[\\"index_inputs_manifest_sha256\\"]\\n                or inputs[\\"candidate_manifest_sha256\\"] != item[\\"candidate_manifest_sha256\\"]\\n                or inputs[\\"input_count\\"] != item[\\"unique_dense_inputs\\"] or len(inputs[\\"parts\\"]) != item[\\"input_parts\\"]):\\n            raise ValueError(\\"Frozen input manifest binding changed.\\")\\n        cursor = 0\\n        for part in inputs[\\"parts\\"]:\\n            if (part[\\"start\\"] != cursor or part[\\"rows\\"] != min(4096, inputs[\\"input_count\\"]-cursor)\\n                    or part[\\"signature\\"] != inputs[\\"signature\\"] or not part[\\"path\\"].endswith(\\".parquet\\")):\\n                raise ValueError(\\"Frozen input part range changed.\\")\\n            bound(build / \\"index_inputs\\", part[\\"path\\"])\\n            sources.append({\\"ordinal\\": ordinal, \\"build\\": item, \\"inputs\\": inputs, \\"part\\": part})\\n            cursor += part[\\"rows\\"]\\n            ordinal += 1\\n        if cursor != inputs[\\"input_count\\"]:\\n            raise ValueError(\\"Frozen input coverage changed.\\")\\n    if len(sources) != batch[\\"input_parts\\"]:\\n        raise ValueError(\\"Batch part coverage changed.\\")\\n    return root, batch, sources\\n\\n\\ndef embedding_contract(data_root, checkout):\\n    \\"\\"\\"Metadata-only: bind exactly to the existing paid BGE/Qwen producers.\\"\\"\\"\\n    data_root, checkout = Path(data_root), Path(checkout)\\n    runtime_lock = checked(read_json(data_root / \\"retrieval_embedding_runtime_lock.json\\"))\\n    runtime = runtime_lock[\\"runtime\\"]\\n    validate_runtime_lock(runtime)\\n    config = read_json(checkout / \\"configs/retrieval_full.json\\")\\n    active = read_json(data_root / \\"active_data_candidate.json\\")\\n    base = data_root / resources_relative(active, config)\\n    for name, checksum in runtime_lock[\\"model_code\\"].items():\\n        if name not in (\\"retrieval_models.py\\", \\"qwen_models.py\\") or sha256_file(checkout / \\"src/vietmedbridge\\" / name) != checksum:\\n            raise ValueError(\\"Pinned model producer differs from the existing vector cache.\\")\\n    job = {\\"config\\": config, \\"runtime\\": runtime, \\"model_code\\": runtime_lock[\\"model_code\\"]}\\n    identities = {}\\n    for family in FAMILIES:\\n        saved = read_json(base / family / \\"config.json\\")\\n        validate_encoder(job, family, saved[\\"encoder\\"], saved[\\"dimension\\"])\\n        if saved.get(\\"format\\") != \\"content-reference-v1\\":\\n            raise ValueError(\\"Existing embedding producer format changed.\\")\\n        encoder = saved[\\"encoder\\"]\\n        count = encoder.get(\\"parameters_before_quantization\\", encoder.get(\\"parameters\\"))\\n        if type(count) is not int or not 0 < count <= 15_000_000_000:\\n            raise ValueError(\\"Embedding model parameter budget is not verified.\\")\\n        identities[family] = {\\"encoder\\": encoder, \\"dimension\\": saved[\\"dimension\\"]}\\n    return config, runtime, identities, active[\\"candidate_manifest_sha256\\"]\\n\\n\\nclass LazyEncoder:\\n    \\"\\"\\"No weights until a cache lookup finds at least one missing text.\\"\\"\\"\\n    def __init__(self, expected, factory):\\n        self.identity, self.dimension = expected[\\"encoder\\"], expected[\\"dimension\\"]\\n        self.factory, self.owner, self.encoder = factory, None, None\\n\\n    def encode(self, texts, *, batch_size):\\n        if self.encoder is None:\\n            self.owner = self.factory()\\n            self.encoder = self.owner.for_role(\\"corpus\\") if hasattr(self.owner, \\"for_role\\") else self.owner\\n            if self.encoder.identity != self.identity or self.encoder.dimension != self.dimension:\\n                self.close()\\n                raise ValueError(\\"GPU model identity differs from the paid content cache; no new cache namespace created.\\")\\n            from vietmedbridge.qwen_execution import configure_embedding_execution\\n            configure_embedding_execution(self.encoder)\\n        return self.encoder.encode(texts, batch_size=batch_size)\\n\\n    def close(self):\\n        if self.owner is not None:\\n            self.owner.close()\\n            self.owner = self.encoder = None\\n\\n\\nclass NewDataEmbeddings:\\n    def __init__(self, project, work_dir, identities, *, team_size=1, worker_id=0):\\n        if type(team_size) is not int or team_size < 1 or type(worker_id) is not int or not 0 <= worker_id < team_size:\\n            raise ValueError(\\"Invalid GPU team/worker assignment.\\")\\n        self.project, self.work = Path(project).resolve(), Path(work_dir).resolve()\\n        self.root, self.batch, self.sources = load_new_batch(self.project)\\n        self.data = self.project / \\"data\\"\\n        self.output = self.root / \\"embeddings_v1\\"\\n        self.identities, self.team_size, self.worker_id = identities, team_size, worker_id\\n        self.work.mkdir(parents=True, exist_ok=True)\\n        self.phase = \\"manifest preflight\\"\\n        binding = {\\"schema_version\\": 1, \\"batch\\": self.batch[\\"manifest_sha256\\"], \\"models\\": identities,\\n                   \\"format\\": \\"NEW_DATA_CONTENT_EMBEDDINGS_V1\\"}\\n        path = self.output / \\"contract.json\\"\\n        if path.is_file() and checked(read_json(path)) != seal(binding):\\n            raise ValueError(\\"New-data embedding contract changed; retain its checkpoints.\\")\\n        atomic_json(path, seal(binding))\\n        self.binding = digest_json(binding)\\n\\n    def _key(self, source):\\n        return source[\\"build\\"][\\"build_run\\"] + \\":\\" + str(source[\\"part\\"][\\"start\\"])\\n\\n    def _view(self, family, source):\\n        inputs = source[\\"inputs\\"]\\n        expected = self.identities[family]\\n        identity = {\\"input_manifest_sha256\\": inputs[\\"manifest_sha256\\"], \\"input_count\\": inputs[\\"input_count\\"],\\n                    \\"encoder\\": expected[\\"encoder\\"], \\"dimension\\": expected[\\"dimension\\"], \\"format\\": \\"content-reference-v1\\"}\\n        return self.output / family / source[\\"build\\"][\\"build_run\\"], identity\\n\\n    def _valid_part(self, family, source, saved):\\n        checked(saved)\\n        _, identity = self._view(family, source)\\n        part = source[\\"part\\"]\\n        expected = {\\"signature\\": digest_json(identity), \\"start\\": part[\\"start\\"], \\"rows\\": part[\\"rows\\"],\\n                    \\"input_sha256\\": part[\\"sha256\\"], \\"path\\": f\\"map-{part[\'start\']:012d}.parquet\\"}\\n        if any(saved.get(k) != v for k, v in expected.items()):\\n            raise ValueError(\\"Embedding part receipt differs from frozen input/model policy.\\")\\n        return saved\\n\\n    def _journals(self, family):\\n        expected = {self._key(s): s for s in self.sources}\\n        merged = {}\\n        directory = self.output / family / \\"progress\\"\\n        for path in sorted(directory.glob(\\"worker-*.json\\")):\\n            saved = checked(read_json(path))\\n            if saved.get(\\"binding\\") != self.binding or saved.get(\\"family\\") != family:\\n                raise ValueError(\\"GPU worker journal belongs to another batch/model.\\")\\n            for name, record in saved[\\"completed\\"].items():\\n                if name not in expected:\\n                    raise ValueError(\\"GPU journal contains an unknown frozen input part.\\")\\n                self._valid_part(family, expected[name], record)\\n                if name in merged and merged[name] != record:\\n                    raise ValueError(\\"GPU workers published conflicting immutable map receipts.\\")\\n                merged[name] = record\\n        return merged\\n\\n    def _save(self, family, completed):\\n        payload = {\\"binding\\": self.binding, \\"family\\": family, \\"team_size\\": self.team_size,\\n                   \\"worker_id\\": self.worker_id, \\"completed\\": completed, \\"updated_at\\": utc_now()}\\n        atomic_json(self.output / family / \\"progress\\" / f\\"worker-{self.worker_id}-of-{self.team_size}.json\\", seal(payload))\\n\\n    def _seal_views(self, family, completed):\\n        grouped = {}\\n        for source in self.sources:\\n            grouped.setdefault(source[\\"build\\"][\\"build_run\\"], []).append(source)\\n        for sources in grouped.values():\\n            root, identity = self._view(family, sources[0])\\n            if all(self._key(s) in completed for s in sources):\\n                path = root / \\"embeddings.json\\"\\n                value = seal(identity | {\\"signature\\": digest_json(identity), \\"state\\": \\"COMPLETE\\",\\n                    \\"parts\\": [completed[self._key(s)] for s in sources]})\\n                if path.is_file() and checked(read_json(path)) != value:\\n                    raise ValueError(\\"Completed embedding view changed.\\")\\n                if not path.is_file():\\n                    atomic_json(path, value)\\n", "new_data_cache_lookup.py": "\\"\\"\\"Released resumable receipt lookup; model producers are unchanged.\\"\\"\\"\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport gzip\\nfrom pathlib import Path, PurePosixPath\\nimport shutil\\nimport sqlite3\\nimport time\\nfrom vietmedbridge.artifacts import (atomic_json, digest_json, local_workspace, publish_file,\\n    read_json, sha256_file, utc_now, verify_file)\\nfrom vietmedbridge.content_embeddings import ContentBlocks as NativeContentBlocks, checked, seal\\nfrom vietmedbridge.scale_vectors import bound\\ntry:\\n    from .new_data_drive_api import LookupPublishUnavailable\\nexcept ImportError:\\n    from new_data_drive_api import LookupPublishUnavailable\\n\\n\\nclass ResumableContentBlocks(NativeContentBlocks):\\n    \\"\\"\\"Retain validated receipt lookups on local disk across Drive recovery.\\n\\n    The native content key, encoder identity and vector payload validation stay\\n    unchanged. A compressed lookup is an optimization, never a coverage receipt.\\n    \\"\\"\\"\\n    def __init__(self, data_root, encoder, work_dir, *, progress=print, receipt_api=None, inventory=True):\\n        self.receipt_api = receipt_api\\n        if receipt_api is not None and Path(data_root).absolute() not in (receipt_api.requested_root, receipt_api.data_root):\\n            raise ValueError(\'Receipt API root differs from the embedding data root.\')\\n        self.root = receipt_api.data_root if receipt_api is not None else Path(data_root).resolve()\\n        self.encoder = encoder\\n        self.identity = {\'encoder\': encoder.identity, \'dimension\': encoder.dimension,\\n            \'format\': \'content-float32-l2-v1\'}\\n        self.directory = self.root / \'model_cache\' / digest_json(self.identity)[:24]\\n        if receipt_api is None:\\n            self.directory.mkdir(parents=True, exist_ok=True)\\n        self.progress = progress\\n        self.binding = {\'schema_version\': 1, \'data_root\': str(self.root), \'identity\': self.identity}\\n        self.lookup = Path(work_dir) / \'receipt_lookups\' / digest_json(self.binding)[:16]\\n        self.lookup.mkdir(parents=True, exist_ok=True)\\n        self.database = self.lookup / \'lookup.sqlite\'\\n        self.snapshot_binding = (dict(schema_version=2, data_folder_id=receipt_api.folder_id, identity=self.identity)\\n            if receipt_api is not None and hasattr(receipt_api, \'folder_id\') else self.binding)\\n        self.remote = self.root / \'retrieval/content_lookups\' / digest_json(self.snapshot_binding)[:16]\\n        self.legacy_remote = self.root / \'retrieval/content_lookups\' / digest_json(self.binding)[:16]\\n        self.temporary = local_workspace(work_dir)\\n        self.local = Path(self.temporary.__enter__())\\n        self.con = None\\n        self._snapshot_dirty = False\\n        self._snapshot_busy = False\\n        self._next_snapshot_count = 256\\n        self._receipt_count = 0\\n        try:\\n            if not self.database.exists():\\n                if not inventory:\\n                    raise FileNotFoundError(\'No local lookup to save; no receipt scan or model inference was started.\')\\n                self._restore()\\n            self.con = sqlite3.connect(self.database)\\n            self.con.execute(\'PRAGMA cache_size=-32768\')\\n            self.con.execute(\'PRAGMA synchronous=FULL\')\\n            self.con.execute(\'CREATE TABLE IF NOT EXISTS metadata(value TEXT NOT NULL)\')\\n            row = self.con.execute(\'SELECT value FROM metadata\').fetchone()\\n            signature = digest_json(self.binding)\\n            if row is not None and row[0] != signature:\\n                raise ValueError(\'Receipt lookup model/root binding changed.\')\\n            if row is None:\\n                self.con.execute(\'INSERT INTO metadata VALUES (?)\', [signature])\\n            self.con.execute(\'CREATE TABLE IF NOT EXISTS vectors(key TEXT PRIMARY KEY,path TEXT,sha TEXT,row INTEGER)\')\\n            self.con.execute(\'CREATE TABLE IF NOT EXISTS receipts(name TEXT PRIMARY KEY,digest TEXT NOT NULL)\')\\n            self.con.commit()\\n            known = {row[0] for row in self.con.execute(\'SELECT name FROM receipts\')}\\n            self._receipt_count = len(known)\\n            self._next_snapshot_count = max(256, len(known)*2)\\n            if not inventory:\\n                self._snapshot_dirty = True\\n                self._snapshot()\\n                return\\n            self.progress(f\'BGE receipt inventory: retained {len(known):,} validated receipts; listing new names\')\\n            markers = (receipt_api.receipts(self.identity) if receipt_api is not None\\n                else sorted(self.directory.glob(\'*.done.json\')))\\n            pending = [path for path in markers if path.name not in known]\\n            self.progress(f\'BGE receipt inventory: {len(markers)-len(pending):,}/{len(markers):,} reused\'\\n                f\' | {len(pending):,} unread | {\\"API\\" if receipt_api is not None else \\"mount\\"} transport | commit each receipt\')\\n            self._read_pending(pending, len(markers)-len(pending), len(markers))\\n            self.progress(f\'BGE receipt inventory ready: {len(markers):,} receipts | native vector SHA checked on use\')\\n            if receipt_api is not None:\\n                if pending:\\n                    self._snapshot_dirty = True\\n                if hasattr(receipt_api, \'publish_lookup\') and not self._snapshot_dirty:\\n                    saved = receipt_api.relative_json((self.remote / \'current.json\').relative_to(self.root).as_posix())\\n                    if saved is None:\\n                        self._snapshot_dirty = True\\n                    else:\\n                        checked(saved)\\n                        if saved.get(\'binding\') != self.snapshot_binding:\\n                            raise ValueError(\'Published lookup producer/folder changed.\')\\n                        self._snapshot_dirty = saved[\'receipts\'] < self._receipt_count\\n                if self._snapshot_dirty:\\n                    self._snapshot()\\n                self.progress(\'BGE API receipt lookup ready; no mount I/O for inventory\')\\n            elif pending or not (self.remote / \'current.json\').is_file():\\n                self._snapshot()\\n        except BaseException:\\n            self.close()\\n            raise\\n\\n    def _read_pending(self, pending, reused, total):\\n        # Keep at most two reads in flight: pool.map queued the whole directory\\n        # and discarded its temporary SQLite lookup after any mount failure.\\n        readers = 4 if self.receipt_api is not None else 2\\n        pool = ThreadPoolExecutor(max_workers=readers)\\n        iterator, futures = iter(pending), []\\n        count, last_log = 0, time.monotonic()\\n        def refill():\\n            while len(futures) < readers:\\n                path = next(iterator, None)\\n                if path is None:\\n                    break\\n                futures.append((path, pool.submit(lambda p: checked(\\n                    p.read_json() if self.receipt_api is not None else read_json(p)), path)))\\n        try:\\n            refill()\\n            while futures:\\n                # Register in native filename order, preserving the first\\n                # existing vector reference when blocks contain overlapping text.\\n                path, future = futures.pop(0)\\n                block = future.result()\\n                if path.name != \'block-\' + digest_json(block[\'keys\']) + \'.done.json\':\\n                    raise ValueError(\'Content receipt name/key binding changed.\')\\n                self.register(block)\\n                count += 1\\n                if count % 32 == 0 or time.monotonic()-last_log >= 15 or count == len(pending):\\n                    self.progress(f\'BGE receipt inventory: {reused+count:,}/{total:,}\'\\n                        f\' | reused {reused:,}, newly indexed {count:,} | local checkpoint committed\')\\n                    last_log = time.monotonic()\\n                refill()\\n        except BaseException:\\n            pool.shutdown(wait=False, cancel_futures=True)\\n            raise\\n        else:\\n            pool.shutdown(wait=True)\\n\\n    def register(self, block):\\n        checked(block)\\n        marker = \'block-\' + digest_json(block[\'keys\']) + \'.done.json\'\\n        old = self.con.execute(\'SELECT digest FROM receipts WHERE name=?\', [marker]).fetchone()\\n        if old is not None and old[0] != block[\'manifest_sha256\']:\\n            raise ValueError(\'Immutable content receipt changed in the retained lookup.\')\\n        try:\\n            # Keys and their receipt commit together; interrupted reads never\\n            # advance the inventory or erase an earlier committed prefix.\\n            if self.receipt_api is None:\\n                super().register(block)\\n            else:\\n                if block[\'identity\'] != self.identity or len(block[\'keys\']) != block[\'rows\']:\\n                    raise ValueError(\'Content vector producer/key mismatch.\')\\n                relative = PurePosixPath(block[\'path\'])\\n                if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in block[\'path\'] or not relative.parts:\\n                    raise ValueError(\'API content reference escapes its data root.\')\\n                # No filesystem resolution through FUSE during API inventory.\\n                # Native vector readers still resolve the actual path and check\\n                # its SHA256/shape before consuming the vector payload.\\n                self.con.executemany(\'INSERT OR IGNORE INTO vectors VALUES (?,?,?,?)\',\\n                    [(key,block[\'path\'],block[\'sha256\'],i) for i,key in enumerate(block[\'keys\'])])\\n            self.con.execute(\'INSERT OR IGNORE INTO receipts VALUES (?,?)\', [marker, block[\'manifest_sha256\']])\\n            self.con.commit()\\n            if old is None:\\n                self._receipt_count += 1\\n                self._snapshot_dirty = True\\n        except BaseException:\\n            self.con.rollback()\\n            raise\\n        if (self.receipt_api is not None and self._receipt_count >= self._next_snapshot_count\\n            and hasattr(self.receipt_api, \'publish_lookup\')):\\n            self._next_snapshot_count = max(256, self._receipt_count*2)\\n            self._snapshot()\\n\\n    def _restore(self):\\n        remote = self.remote\\n        marker = remote / \'current.json\'\\n        if self.receipt_api is None:\\n            if not marker.is_file():\\n                return\\n            saved = checked(read_json(marker))\\n        else:\\n            value = self.receipt_api.relative_json(marker.relative_to(self.root).as_posix())\\n            if value is None and self.legacy_remote != self.remote:\\n                remote = self.legacy_remote\\n                value = self.receipt_api.relative_json((remote / \'current.json\').relative_to(self.root).as_posix())\\n            if value is None:\\n                return\\n            saved = checked(value)\\n        if saved.get(\'binding\') != (self.snapshot_binding if remote == self.remote else self.binding):\\n            raise ValueError(\'Published receipt lookup producer/root changed.\')\\n        relative = PurePosixPath(saved[\'path\'])\\n        if relative.is_absolute() or len(relative.parts) != 1 or \'\\\\\\\\\' in saved[\'path\']:\\n            raise ValueError(\'Published receipt snapshot path escapes its directory.\')\\n        self.progress(\'BGE receipt inventory: restore one compressed lookup; no old receipt JSON scan\')\\n        local = self.local / \'lookup.sqlite.gz\'\\n        if self.receipt_api is None:\\n            shutil.copyfile(bound(remote, saved[\'path\']), local)\\n        else:\\n            source = self.receipt_api.relative((remote / saved[\'path\']).relative_to(self.root).as_posix())\\n            if source is None:\\n                raise ValueError(\'Published API receipt lookup payload is missing.\')\\n            self.receipt_api.download(source, local, expected_sha256=saved[\'sha256\'])\\n        verify_file(local, saved[\'sha256\'])\\n        stage = self.lookup / \'restore.sqlite\'\\n        try:\\n            with gzip.open(local, \'rb\') as src, stage.open(\'wb\') as dst:\\n                shutil.copyfileobj(src, dst, length=1024*1024)\\n            verify_file(stage, saved[\'sqlite_sha256\'])\\n            # A shared folder has the same identity on another account even when\\n            # Colab exposes it through a different MyDrive/shortcut path.\\n            old_binding = saved.get(\'lookup_binding\', saved[\'binding\'])\\n            restored = sqlite3.connect(stage)\\n            try:\\n                rows = restored.execute(\'SELECT value FROM metadata\').fetchall()\\n                if rows != [(digest_json(old_binding),)] or old_binding.get(\'identity\') != self.identity:\\n                    raise ValueError(\'Restored lookup SQLite binding changed.\')\\n                if restored.execute(\'SELECT count(*) FROM receipts\').fetchone()[0] != saved[\'receipts\']:\\n                    raise ValueError(\'Restored lookup receipt count changed.\')\\n                if old_binding != self.binding:\\n                    restored.execute(\'UPDATE metadata SET value=?\', [digest_json(self.binding)])\\n                    restored.commit()\\n            finally:\\n                restored.close()\\n            stage.replace(self.database)\\n        finally:\\n            stage.unlink(missing_ok=True)\\n\\n    def _snapshot(self):\\n        if self._snapshot_busy:\\n            return\\n        if self.receipt_api is not None and not hasattr(self.receipt_api, \'publish_lookup\'):\\n            return  # Compatibility with older read-only transports; local prefix remains usable.\\n        self._snapshot_busy = True\\n        self.progress(\'BGE receipt inventory: save compressed lookup for future runtimes\')\\n        copy = self.local / \'lookup-backup.sqlite\'\\n        payload = self.local / \'lookup.sqlite.gz\'\\n        try:\\n            backup = sqlite3.connect(copy)\\n            try:\\n                self.con.backup(backup)\\n            finally:\\n                backup.close()\\n            checksum = sha256_file(copy)\\n            with copy.open(\'rb\') as src, gzip.open(payload, \'wb\', compresslevel=1) as dst:\\n                shutil.copyfileobj(src, dst, length=1024*1024)\\n            target = self.remote / (\'lookup-\' + sha256_file(payload) + \'.sqlite.gz\')\\n            record = seal({\'binding\': self.snapshot_binding, \'lookup_binding\': self.binding, \'path\': target.name,\\n                \'sha256\': sha256_file(payload), \'sqlite_sha256\': checksum,\\n                \'receipts\': self.con.execute(\'SELECT count(*) FROM receipts\').fetchone()[0],\\n                \'updated_at\': utc_now(),\\n                \'scope\': \'Receipt lookup only; vector payloads remain subject to native SHA/shape checks\'})\\n            if self.receipt_api is None:\\n                publish_file(payload, target)\\n                atomic_json(self.remote / \'current.json\', record)\\n            else:\\n                self.receipt_api.publish_lookup((self.remote / \'current.json\').relative_to(self.root).as_posix(),\\n                    payload, record)\\n            self._snapshot_dirty = False\\n            self.progress(f\\"BGE receipt lookup SAVED_ON_DRIVE: {record[\'receipts\']:,} receipts; next runtime restores this lookup\\")\\n        except (OSError, LookupPublishUnavailable) as error:\\n            # Optional acceleration only: the local committed database survives,\\n            # and native reads/writes still surface actual mount/space failures.\\n            self.progress(f\'BGE receipt lookup snapshot deferred: {type(error).__name__}; local checkpoint retained\')\\n        finally:\\n            self._snapshot_busy = False\\n            copy.unlink(missing_ok=True)\\n            payload.unlink(missing_ok=True)\\n\\n    def close(self):\\n        if self.con is not None:\\n            try:\\n                if self.receipt_api is not None and self._snapshot_dirty:\\n                    self._snapshot()\\n            finally:\\n                self.con.close()\\n                self.con = None\\n        self.temporary.__exit__(None, None, None)\\n", "new_data_drive_api.py": "\\"\\"\\"API receipt reads and narrowly scoped lookup checkpoints, without Drive FUSE.\\"\\"\\"\\nimport copy\\nimport hashlib\\nimport json\\nfrom pathlib import Path, PurePosixPath\\nimport re\\nimport threading\\nimport time\\n\\n\\nclass LookupPublishUnavailable(RuntimeError):\\n    \\"\\"\\"An optional lookup upload failed; paid vector receipts remain untouched.\\"\\"\\"\\n\\n\\nclass RemoteReceipt:\\n    def __init__(self, source, metadata):\\n        self.source, self.metadata = source, metadata\\n        self.name = metadata[\'name\']\\n\\n    def read_json(self):\\n        return self.source.read_json(self.metadata)\\n\\n\\nclass DriveReceiptSource:\\n    endpoint = \'https://www.googleapis.com/drive/v3/files\'\\n\\n    def __init__(self, session_factory, data_root, canonical_root, *,\\n                 data_folder_id=\'1Ez1S9jk6FO_puxC8r2vPSc0LJbuOkkdj\', report=print):\\n        if not re.fullmatch(r\'[A-Za-z0-9_-]+\', data_folder_id):\\n            raise ValueError(\'Invalid Drive data folder ID.\')\\n        self.session_factory, self.sessions = session_factory, threading.local()\\n        self.requested_root, self.data_root = Path(data_root).absolute(), Path(canonical_root)\\n        self.folder_id, self.report, self.folders = data_folder_id, report, {}\\n        self.payload_directories = {}\\n\\n    def directory_files(self, relative_directory, *, refresh=False):\\n        \\"\\"\\"One paginated metadata listing, rather than one lookup per payload.\\"\\"\\"\\n        relative = PurePosixPath(relative_directory)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in relative_directory or not relative.parts:\\n            raise ValueError(\'Invalid payload directory.\')\\n        if not refresh and relative_directory in self.payload_directories:\\n            return self.payload_directories[relative_directory]\\n        parent = self.folder_id\\n        for part in relative.parts:\\n            if (parent, part) not in self.folders:\\n                value = self.child(parent, part, folder=True)\\n                if value is None:\\n                    return {}\\n                self.folders[parent, part] = value[\'id\']\\n            parent = self.folders[parent, part]\\n        result = {}\\n        for value in self._list(f\\"\'{parent}\' in parents and trashed=false\\"):\\n            if value[\'name\'] in result:\\n                raise ValueError(\'Duplicate payload names in Drive directory.\')\\n            result[value[\'name\']] = value\\n        self.payload_directories[relative_directory] = result\\n        return result\\n\\n    def payload_metadata(self, relative_path):\\n        relative = PurePosixPath(relative_path)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in relative_path or len(relative.parts) < 2:\\n            raise ValueError(\'Invalid immutable payload path.\')\\n        directory = relative.parent.as_posix()\\n        result = self.directory_files(directory).get(relative.name)\\n        if result is None:\\n            result = self.directory_files(directory, refresh=True).get(relative.name)\\n        if result is None or \'size\' not in result:\\n            raise ValueError(\'Expected immutable payload is missing from Drive.\')\\n        return result\\n\\n    def _session(self):\\n        if not hasattr(self.sessions, \'session\'):\\n            self.sessions.session = self.session_factory()\\n        return self.sessions.session\\n\\n    def _get(self, url, params, *, stream=False):\\n        import requests\\n        session = self._session()\\n        for attempt in range(4):\\n            try:\\n                response = session.get(url, params=params, stream=stream, timeout=(10,60))\\n            except requests.RequestException:\\n                if attempt == 3:\\n                    raise\\n            else:\\n                if response.status_code not in (429,500,502,503,504) or attempt == 3:\\n                    response.raise_for_status()\\n                    return response\\n                response.close()\\n            self.report(f\'Drive receipt API: transient request failure; retry {attempt+1}/3\')\\n            time.sleep(2**attempt)\\n\\n    def _list(self, query):\\n        token, seen = None, set()\\n        while True:\\n            params = {\'q\': query, \'pageSize\': 1000, \'supportsAllDrives\': \'true\',\\n                \'includeItemsFromAllDrives\': \'true\',\\n                \'fields\': \'nextPageToken,incompleteSearch,files(id,name,mimeType,size,md5Checksum)\'}\\n            if token:\\n                params[\'pageToken\'] = token\\n            with self._get(self.endpoint, params) as response:\\n                page = response.json()\\n            if page.get(\'incompleteSearch\'):\\n                raise ValueError(\'Drive receipt listing is incomplete; refusing partial inventory.\')\\n            yield from page.get(\'files\', [])\\n            token = page.get(\'nextPageToken\')\\n            if not token:\\n                return\\n            if token in seen:\\n                raise ValueError(\'Drive receipt pagination repeated; inventory is incomplete.\')\\n            seen.add(token)\\n\\n    def child(self, parent, name, *, folder=False):\\n        safe = name.replace(\'\\\\\\\\\', \'\\\\\\\\\\\\\\\\\').replace(\\"\'\\", \\"\\\\\\\\\'\\")\\n        query = f\\"\'{parent}\' in parents and trashed=false and name=\'{safe}\'\\"\\n        if folder:\\n            query += \\" and mimeType=\'application/vnd.google-apps.folder\'\\"\\n        values = list(self._list(query))\\n        if len(values) > 1:\\n            raise ValueError(\'Duplicate Drive path names; refusing ambiguous receipt source.\')\\n        return values[0] if values else None\\n\\n    def relative(self, name):\\n        relative = PurePosixPath(name)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in name or not relative.parts:\\n            raise ValueError(\'Invalid relative Drive receipt path.\')\\n        parent = self.folder_id\\n        for part in relative.parts[:-1]:\\n            key = parent, part\\n            if key not in self.folders:\\n                value = self.child(parent, part, folder=True)\\n                if value is None:\\n                    return None\\n                self.folders[key] = value[\'id\']\\n            parent = self.folders[key]\\n        return self.child(parent, relative.name)\\n\\n    def download(self, metadata, target, *, expected_sha256=None, max_bytes=None):\\n        size = int(metadata[\'size\'])\\n        if max_bytes is not None and size > max_bytes:\\n            raise ValueError(\'Receipt exceeds the bounded JSON size.\')\\n        sha, md5, count = hashlib.sha256(), hashlib.md5(), 0\\n        with self._get(self.endpoint+\'/\'+metadata[\'id\'], {\'alt\':\'media\',\'supportsAllDrives\':\'true\'}, stream=True) as response:\\n            with Path(target).open(\'wb\') as output:\\n                for chunk in response.iter_content(chunk_size=1024*1024):\\n                    count += len(chunk)\\n                    if count > size or (max_bytes is not None and count > max_bytes):\\n                        raise ValueError(\'Drive receipt payload exceeds its declared size.\')\\n                    sha.update(chunk)\\n                    md5.update(chunk)\\n                    output.write(chunk)\\n        if (count != size or (metadata.get(\'md5Checksum\') and md5.hexdigest() != metadata[\'md5Checksum\'])\\n            or (expected_sha256 and sha.hexdigest() != expected_sha256)):\\n            raise ValueError(\'Drive receipt size/checksum mismatch.\')\\n\\n    def read_json(self, metadata):\\n        import tempfile\\n        with tempfile.TemporaryDirectory(prefix=\'vmb-api-json-\') as directory:\\n            path = Path(directory) / \'receipt.json\'\\n            self.download(metadata, path, max_bytes=16*1024*1024)\\n            return json.loads(path.read_text(encoding=\'utf-8\'))\\n\\n    def relative_json(self, path):\\n        metadata = self.relative(path)\\n        return None if metadata is None else self.read_json(metadata)\\n\\n    def verify_candidate(self, expected):\\n        active = self.relative_json(\'active_data_candidate.json\')\\n        if active is None or active.get(\'candidate_manifest_sha256\') != expected:\\n            raise ValueError(\'Drive API data folder differs from the expected frozen candidate.\')\\n        candidate = self.relative_json(\'processed/\'+active[\'build_run\']+\'/\'+active[\'candidate_name\'])\\n        from vietmedbridge.artifacts import digest_json\\n        if (candidate is None or candidate.get(\'candidate_manifest_sha256\') != expected\\n            or digest_json({k:v for k,v in candidate.items() if k != \'candidate_manifest_sha256\'}) != expected):\\n            raise ValueError(\'Drive API candidate digest changed.\')\\n        self.active, self.candidate = active, candidate\\n\\n    def receipts(self, identity):\\n        from vietmedbridge.artifacts import digest_json\\n        # Resolve the folder itself without relying on any receipt being present.\\n        cache = self.child(self.folder_id, \'model_cache\', folder=True)\\n        folder = self.child(cache[\'id\'], digest_json(identity)[:24], folder=True) if cache else None\\n        if folder is None:\\n            return []\\n        result, names, payloads = [], set(), {}\\n        # Drive name-contains is prefix matching, not a suffix predicate. List\\n        # the complete producer folder then filter receipt names locally.\\n        query = f\\"\'{folder[\'id\']}\' in parents and trashed=false\\"\\n        for value in self._list(query):\\n            if value[\'name\'] in payloads:\\n                raise ValueError(\'Duplicate names in content cache directory.\')\\n            payloads[value[\'name\']] = value\\n            if not value[\'name\'].endswith(\'.done.json\'):\\n                continue\\n            if (not re.fullmatch(r\'block-[0-9a-f]{64}\\\\.done\\\\.json\', value[\'name\'])\\n                or value[\'name\'] in names or \'size\' not in value):\\n                raise ValueError(\'Invalid or duplicate API receipt filename.\')\\n            names.add(value[\'name\'])\\n            result.append(RemoteReceipt(self, value))\\n        self.payload_directories[\'model_cache/\'+digest_json(identity)[:24]] = payloads\\n        self.report(f\'Drive receipt API: complete listing, {len(result):,} receipts; no mount reads\')\\n        return sorted(result, key=lambda value: value.name)\\n\\n    def publish_cache_checkpoint(self, relative_marker, payload, marker):\\n        \\"\\"\\"Only mapping bundles and exact ANN training samples may be published here.\\"\\"\\"\\n        from vietmedbridge.artifacts import digest_json\\n        binding = marker.get(\'binding\', {})\\n        source = binding.get(\'source_manifest_sha256\', \'\')\\n        kind = binding.get(\'kind\')\\n        if kind == \'mapping-bundle-v1\':\\n            directory, prefix, suffix = \'retrieval/mapping_bundles/\'+source, \'maps-\', \'.tar.gz\'\\n        elif kind == \'index-sample-v1\' and re.fullmatch(r\'[0-9a-f]{64}\', binding.get(\'positions_sha256\', \'\')):\\n            directory = \'retrieval/index_samples/\'+source+\'/\'+binding[\'positions_sha256\']\\n            prefix, suffix = \'sample-\', \'.npy\'\\n        else:\\n            raise ValueError(\'Unsupported cache checkpoint kind.\')\\n        if (binding.get(\'data_folder_id\') != self.folder_id or binding.get(\'schema_version\') != 1\\n            or not re.fullmatch(r\'[0-9a-f]{64}\', source)\\n            or relative_marker != directory+\'/current.json\'\\n            or not re.fullmatch(r\'[0-9a-f]{64}\', marker.get(\'sha256\', \'\'))\\n            or marker.get(\'path\') != prefix+marker[\'sha256\']+suffix\\n            or marker.get(\'manifest_sha256\') != digest_json({k:v for k,v in marker.items() if k != \'manifest_sha256\'})):\\n            raise ValueError(\'Cache checkpoint escapes its declared source/folder binding.\')\\n        sha, md5 = hashlib.sha256(), hashlib.md5()\\n        with Path(payload).open(\'rb\') as stream:\\n            for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\\n                sha.update(chunk); md5.update(chunk)\\n        if sha.hexdigest() != marker[\'sha256\']:\\n            raise ValueError(\'Cache checkpoint payload changed before upload.\')\\n        try:\\n            return self._publish_lookup_files(directory, Path(payload), marker, md5.hexdigest())\\n        except ValueError:\\n            raise\\n        except Exception as error:\\n            raise LookupPublishUnavailable(type(error).__name__) from error\\n\\n    def publish_lookup(self, relative_marker, payload, marker):\\n        \\"\\"\\"Upload a checksum-addressed lookup, then its pointer. Never write vector/source files.\\"\\"\\"\\n        from vietmedbridge.artifacts import digest_json\\n        body = {k:v for k,v in marker.items() if k != \'manifest_sha256\'}\\n        binding = marker.get(\'binding\', {})\\n        expected_directory = \'retrieval/content_lookups/\' + digest_json(binding)[:16]\\n        if (relative_marker != expected_directory + \'/current.json\'\\n            or binding.get(\'schema_version\') != 2 or binding.get(\'data_folder_id\') != self.folder_id\\n            or marker.get(\'manifest_sha256\') != digest_json(body)\\n            or marker.get(\'path\') != \'lookup-\' + marker.get(\'sha256\', \'\') + \'.sqlite.gz\'\\n            or not re.fullmatch(r\'lookup-[0-9a-f]{64}\\\\.sqlite\\\\.gz\', marker.get(\'path\', \'\'))):\\n            raise ValueError(\'Lookup upload is outside its bound checkpoint directory.\')\\n        sha, md5 = hashlib.sha256(), hashlib.md5()\\n        with Path(payload).open(\'rb\') as stream:\\n            for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\\n                sha.update(chunk)\\n                md5.update(chunk)\\n        if sha.hexdigest() != marker[\'sha256\']:\\n            raise ValueError(\'Lookup payload changed before upload.\')\\n        try:\\n            return self._publish_lookup_files(expected_directory, Path(payload), marker, md5.hexdigest())\\n        except ValueError:\\n            raise\\n        except Exception as error:\\n            # Do not include provider messages/URLs: resumable upload URLs are private.\\n            raise LookupPublishUnavailable(type(error).__name__) from error\\n\\n    def _publish_lookup_files(self, directory, payload, marker, md5):\\n        from vietmedbridge.artifacts import digest_json\\n        from googleapiclient.discovery import build\\n        from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload\\n        import io\\n        service = build(\'drive\', \'v3\', credentials=copy.copy(self._session().credentials), cache_discovery=False)\\n        files = service.files()\\n        fields = \'id,name,mimeType,size,md5Checksum\'\\n\\n        def new_id():\\n            return files.generateIds(count=1, space=\'drive\', type=\'files\').execute(num_retries=3)[\'ids\'][0]\\n\\n        def ensure_folder(parent, name):\\n            value = self.child(parent, name, folder=True)\\n            if value is None:\\n                identifier = new_id()\\n                try:\\n                    value = files.create(body={\'id\':identifier, \'name\':name, \'parents\':[parent],\\n                        \'mimeType\':\'application/vnd.google-apps.folder\'}, fields=fields,\\n                        supportsAllDrives=True).execute(num_retries=3)\\n                except Exception:\\n                    # A lost response may follow a successful create. Reconcile the\\n                    # preallocated ID; never retry a folder with a different ID.\\n                    value = files.get(fileId=identifier, fields=fields, supportsAllDrives=True).execute(num_retries=3)\\n                    if value.get(\'name\') != name or value.get(\'mimeType\') != \'application/vnd.google-apps.folder\':\\n                        raise ValueError(\'Created lookup folder identity changed.\')\\n            self.folders[parent, name] = value[\'id\']\\n            return value[\'id\']\\n\\n        parent = self.folder_id\\n        for name in PurePosixPath(directory).parts:\\n            parent = ensure_folder(parent, name)\\n        current = self.child(parent, \'current.json\')\\n        if current is not None:\\n            existing = self.read_json(current)\\n            if (existing.get(\'manifest_sha256\') != digest_json(\\n                    {k:v for k,v in existing.items() if k != \'manifest_sha256\'})\\n                or existing.get(\'binding\') != marker[\'binding\']):\\n                raise ValueError(\'Existing lookup checkpoint binding/digest changed.\')\\n            count_key = \'receipts\' if \'receipts\' in marker else \'parts\'\\n            if existing[count_key] > marker[count_key]:\\n                return existing  # An older local prefix must not replace a newer saved lookup.\\n\\n        def upload(media, name, existing=None):\\n            if existing is None:\\n                request = files.create(body={\'id\':new_id(), \'name\':name, \'parents\':[parent]},\\n                    media_body=media, fields=fields, supportsAllDrives=True)\\n            else:\\n                request = files.update(fileId=existing[\'id\'], media_body=media,\\n                    fields=fields, supportsAllDrives=True)\\n            result = None\\n            while result is None:\\n                _, result = request.next_chunk(num_retries=3)\\n            # Query provider metadata independently before publishing a pointer.\\n            return files.get(fileId=result[\'id\'], fields=fields, supportsAllDrives=True).execute(num_retries=3)\\n\\n        saved = self.child(parent, marker[\'path\'])\\n        if saved is None:\\n            mime = \'application/gzip\' if marker[\'path\'].endswith(\'.gz\') else \'application/octet-stream\'\\n            saved = upload(MediaFileUpload(str(payload), mimetype=mime,\\n                chunksize=8*1024*1024, resumable=True), marker[\'path\'])\\n        if int(saved.get(\'size\', -1)) != payload.stat().st_size or saved.get(\'md5Checksum\') != md5:\\n            raise ValueError(\'Uploaded lookup payload size/checksum mismatch; pointer not advanced.\')\\n        data = json.dumps(marker, ensure_ascii=False, sort_keys=True).encode(\'utf-8\')\\n        saved_marker = upload(MediaIoBaseUpload(io.BytesIO(data), mimetype=\'application/json\',\\n            chunksize=256*1024, resumable=True), \'current.json\', current)\\n        if int(saved_marker.get(\'size\', -1)) != len(data) or saved_marker.get(\'md5Checksum\') != hashlib.md5(data).hexdigest():\\n            raise ValueError(\'Uploaded lookup pointer size/checksum mismatch.\')\\n        return marker\\n\\n\\ndef authenticate_drive_receipts(data_root, expected_candidate_sha256):\\n    \\"\\"\\"Authenticate on the main Colab thread; never print/save credential tokens.\\"\\"\\"\\n    from google.colab import auth\\n    import google.auth\\n    from google.auth.transport.requests import AuthorizedSession\\n    root = Path(data_root).resolve()\\n    print(\'Drive receipts: authenticate API; receipts read-only, lookup checkpoints saved in retrieval/content_lookups\', flush=True)\\n    auth.authenticate_user()\\n    credentials, _ = google.auth.default()\\n    client = DriveReceiptSource(lambda: AuthorizedSession(copy.copy(credentials)), data_root, root)\\n    client.verify_candidate(expected_candidate_sha256)\\n    print(\'Drive receipts: API source verified against the same frozen candidate\', flush=True)\\n    return client\\n"}')

for name, source in SOURCES.items():
    (WORK_DIR/name).write_text(source, encoding='utf-8')
import new_data_kaggle as pipeline
from new_data_embeddings import embedding_contract
from new_data_drive_api import DriveReceiptSource
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials, _ = google.auth.default()
config, runtime, identities, anchor = embedding_contract(PROJECT/'data', CHECKOUT)
data = (PROJECT/'data').resolve()
api_source = DriveReceiptSource(lambda: AuthorizedSession(copy.copy(credentials)), data, data)
api_source.verify_candidate(anchor)
print('Runtime GPU khóa từ cache cũ:', runtime, flush=True)
print('Batch: 10 frozen builds; task còn thiếu được chọn riêng cho từng model.', flush=True)


In [ ]:
#@title 3. Xuất gói mới / nhập Outputs rồi xuất lượt còn thiếu
if KAGGLE_NOTEBOOK_OUTPUT.strip():
    from kaggle.api.kaggle_api_extended import KaggleApi
    from vietmedbridge.kaggle_outputs import output_handles
    api = KaggleApi()
    api.authenticate()
    for handle in output_handles(KAGGLE_NOTEBOOK_OUTPUT):
        output = pipeline.download_outputs(api, handle, WORK_DIR, project=PROJECT)
        saved = pipeline.checked(pipeline.read_json(output/'job.json'))
        flow = pipeline.NewDataKaggle(PROJECT, WORK_DIR/'cpu', identities,
            accounts=saved['team']['accounts'], gpu_count=saved['team']['gpu_count'])
        report = flow.install(output, receipt_api=api_source)
        print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)
        KAGGLE_ACCOUNTS = saved['team']['accounts']
flow = pipeline.NewDataKaggle(PROJECT, WORK_DIR/'cpu', identities, accounts=KAGGLE_ACCOUNTS, gpu_count=2)
target, job = flow.export(CHECKOUT, config=config, runtime=runtime, code_commit=lock['commit'], receipt_api=api_source)
if target is None:
    print(json.dumps(job, ensure_ascii=False, indent=2), flush=True)
    if job['state'] == 'WAITING_FOR_KAGGLE_OUTPUTS':
        print('Giữ lượt hiện tại:', 'https://www.kaggle.com/datasets/'+job['dataset_handle'], flush=True)
        print('Nhập Outputs của các acc còn thiếu trước; không xuất lượt chồng lên phiên cũ.', flush=True)
    elif all(v['completed_parts'] == v['total_parts'] for v in job['coverage'].values()):
        print('NEW_DATA_EMBEDDINGS_COMPLETE — BGE và Qwen đủ batch mới. Tiếp theo mới ghép/index.', flush=True)
    else:
        raise RuntimeError('Không còn task nhưng mapping chưa đủ. Giữ checkpoint; chưa xác nhận hoàn tất.')
else:
    pipeline.upload_job(flow, target, job, owner)
    print('Hai acc Add Input CÙNG dataset trên; chạy 04_new_data_account_0 và 04_new_data_account_1.', flush=True)
    print('Nếu KAGGLE_ACCOUNTS=1: chỉ chạy acc 0. Không Add Input gói 700k cũ.', flush=True)
    print('Giới hạn kế hoạch: 16 GB vector mới/acc/lượt; ROUND_COMPLETE chưa có nghĩa cả batch xong.', flush=True)